# CSAM Phase 2 — Multi-Seed Variance Runner (5 Seeds)

**Cognitive Sparse Access Memory (CSAM)** — Phase 2 publishable evaluation.

This notebook runs two evaluations across **5 seeds (42–46)**:
1. **5-Seed Ablation** — 4 strategies × 5 seeds = 20 runs → mean ± std F1 per strategy
2. **5-Seed Threshold Sweep** — CA at 5 thresholds × 5 seeds = 25 runs → mean ± std per threshold

After all seeds complete, the notebook aggregates results and runs **Wilcoxon signed-rank significance tests** (CA vs each baseline).

**Runtime:** CPU (T4 GPU idle — CSAM is CPU + API-bound)  
**API:** Groq free tier (llama-3.1-8b-instant, 2-key rotation)  
**Estimated time:** Ablation ~25–30 min, Sweep ~40–50 min, Total ~65–80 min  
**API calls:** ~1460 ablation + ~2190 sweep = ~3650 total (within 14,400/day free limit)

## 1. Check Runtime Environment

In [ ]:
import os
import platform

print("=" * 60)
print("RUNTIME ENVIRONMENT CHECK")
print("=" * 60)

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
print(f"Running on Colab:  {IN_COLAB}")
print(f"Python:            {platform.python_version()}")
print(f"Platform:          {platform.platform()}")

import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System RAM:        {ram_gb:.1f} GB")
print("=" * 60)

## 2. Install Dependencies

In [ ]:
!pip install -q python-dotenv sentence-transformers hnswlib scipy
print("\n✅ All dependencies installed")

## 3. Clone Repository

In [ ]:
import os
import shutil

REPO_URL = "https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git"
REPO_DIR = "/content/CSAM-IPD-HALH"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone --branch main --depth 1 {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
print(f"\n📂 Working directory: {os.getcwd()}")

# Verify variance_runner.py is present (Phase 2 code)
vr_path = "csam_project/evaluation/variance_runner.py"
if os.path.exists(vr_path):
    print("✅ variance_runner.py found")
else:
    print("❌ variance_runner.py NOT found — make sure Phase 2 code is committed and pushed!")

## 4. Set API Keys & Verify Groq

Load Groq API keys from Colab Secrets (🔑 key icon in left sidebar):
- `GROQ_API_KEY` — primary key
- `GROQ_API_KEY_2` — second key for rate-limit rotation

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    try:
        os.environ["GROQ_API_KEY_2"] = userdata.get("GROQ_API_KEY_2")
    except Exception:
        pass  # Second key is optional
    print("✅ API keys loaded from Colab Secrets")
except Exception:
    if "GROQ_API_KEY" in os.environ:
        print("✅ API keys found in environment")
    else:
        raise RuntimeError(
            "❌ No API keys found! Add GROQ_API_KEY to Colab Secrets "
            "(key icon in left sidebar)"
        )

# Write .env file for CSAM scripts
env_lines = []
for key in ["GROQ_API_KEY", "GROQ_API_KEY_2"]:
    val = os.environ.get(key, "")
    if val:
        env_lines.append(f"{key}={val}")

with open(".env", "w") as f:
    f.write("\n".join(env_lines) + "\n")

print(f"📝 .env written with {len(env_lines)} key(s)")

# Quick Groq connectivity check
import requests
resp = requests.get(
    "https://api.groq.com/openai/v1/models",
    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
    timeout=10,
)
if resp.status_code == 200:
    models = [m["id"] for m in resp.json().get("data", [])]
    print(f"✅ Groq API connected — {len(models)} models available")
    if "llama-3.1-8b-instant" in models:
        print("   ✓ llama-3.1-8b-instant is available")
else:
    print(f"❌ Groq API error: {resp.status_code} — check your key")

## 5. Verify CSAM Imports & Embedding Model

In [ ]:
import sys
import os
import time

os.chdir("/content/CSAM-IPD-HALH")
sys.path.insert(0, os.path.join(os.getcwd(), "csam_project"))

from csam_core.memory_repository import MemoryRepository
from csam_core.forgetting_engine import ConsolidationAwareForgetting
from csam_core.services.embedding import EmbeddingService
from csam_core.services.llm_hosted import HostedLLMService
from evaluation.npc_locomo import BenchmarkGenerator
from evaluation.run_ablation import run_ablation_study
from evaluation.threshold_sweep import run_threshold_sweep
from evaluation.variance_runner import run_variance

print("✅ All CSAM modules imported successfully")

# Verify seed param exists
import inspect
sig = inspect.signature(run_ablation_study)
assert "seed" in sig.parameters, "❌ seed param missing from run_ablation_study!"
print("✅ seed param verified in run_ablation_study")

sig2 = inspect.signature(run_threshold_sweep)
assert "seed" in sig2.parameters, "❌ seed param missing from run_threshold_sweep!"
print("✅ seed param verified in run_threshold_sweep")

# Warm up embedding model
print("\n⏳ Loading embedding model (all-MiniLM-L6-v2)...")
t0 = time.time()
emb = EmbeddingService()
dim = emb.dimension
_ = emb.encode("test")
print(f"✅ Embedding model loaded in {time.time() - t0:.1f}s (dim={dim})")

# Verify BenchmarkGenerator seed propagation
gen42 = BenchmarkGenerator(seed=42)
gen99 = BenchmarkGenerator(seed=99)
ds42 = gen42.generate_benchmark_dataset(num_conversations=1, interactions_per_conversation=5)
ds99 = gen99.generate_benchmark_dataset(num_conversations=1, interactions_per_conversation=5)
texts42 = [i.text for i in ds42[0].interactions]
texts99 = [i.text for i in ds99[0].interactions]
same = (texts42 == texts99)
print(f"\n{'⚠️  seed=42 and seed=99 give SAME dataset (seed NOT propagated)' if same else '✅ seed=42 vs seed=99 datasets differ (seed propagation confirmed)'}")

print("\n" + "=" * 60)
print("ALL CHECKS PASSED — ready to run variance evaluation")
print("=" * 60)

## 6. Configure Run Parameters

Edit the variables below before running. Defaults match the paper configuration.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
SEEDS              = [42, 43, 44, 45, 46]   # 5 seeds for publishable results
MODE               = "both"                  # "ablation", "sweep", or "both"
NUM_CONVERSATIONS  = 5
INTERACTIONS       = 100
FORGET_THRESHOLD   = 80
LLM_MODEL          = "llama-3.1-8b-instant"
OUTPUT_DIR         = "/content/CSAM-IPD-HALH/csam_project/evaluation"
VERBOSE            = False  # Set True to see every Q&A answer (very verbose)

# API calls estimate
qa_per_run = NUM_CONVERSATIONS * 14  # ~14 QA pairs per 5 conversations
ablation_calls = len(SEEDS) * 4 * qa_per_run  # 4 strategies per seed
sweep_calls = len(SEEDS) * 6 * qa_per_run      # 6 evals (1 NF + 5 thresholds)
total_calls = ablation_calls + sweep_calls if MODE == "both" else (ablation_calls if MODE == "ablation" else sweep_calls)

print("=" * 60)
print("RUN CONFIGURATION")
print("=" * 60)
print(f"  Seeds:              {SEEDS}")
print(f"  Mode:               {MODE}")
print(f"  Conversations:      {NUM_CONVERSATIONS}")
print(f"  Interactions:       {INTERACTIONS}")
print(f"  Forget threshold:   {FORGET_THRESHOLD}")
print(f"  Model:              {LLM_MODEL}")
print(f"  Output dir:         {OUTPUT_DIR}")
print(f"  Est. API calls:     ~{total_calls} (limit: 14,400/day)")
print("=" * 60)

## 7. Run 5-Seed Ablation Study

Runs 4 strategies × 5 seeds = **20 total evaluations**.  
Each seed produces `ablation_seed{N}.json`. After all seeds: aggregation + significance tests.

**⏱ Estimated: ~25–30 min**

In [ ]:
import os
import time

os.chdir("/content/CSAM-IPD-HALH")

if MODE in ("ablation", "both"):
    print("🚀 Starting 5-Seed Ablation Study...")
    t_start = time.time()

    for i, seed in enumerate(SEEDS, 1):
        out = f"{OUTPUT_DIR}/ablation_seed{seed}.json"
        print(f"\n[{i}/{len(SEEDS)}] Seed={seed}  → {out}")
        t0 = time.time()

        !python -m csam_project.evaluation.run_ablation \
            --conversations {NUM_CONVERSATIONS} \
            --interactions {INTERACTIONS} \
            --threshold {FORGET_THRESHOLD} \
            --seed {seed} \
            --output {out} \
            --model {LLM_MODEL} \
            {'--quiet' if not VERBOSE else ''}

        elapsed = time.time() - t0
        print(f"  ✅ Seed {seed} done in {elapsed/60:.1f} min")

    total_elapsed = time.time() - t_start
    print(f"\n⏱️  All ablation seeds done in {total_elapsed/60:.1f} min")
else:
    print("⏭️  Skipping ablation (mode=sweep)")

## 8. Run 5-Seed Threshold Sweep

Runs CA at 5 thresholds (50, 80, 100, 120, 150) + No-Forgetting baseline × 5 seeds = **30 total evaluations**.  
Each seed produces `sweep_seed{N}.json`.

**⏱ Estimated: ~40–50 min**

In [ ]:
import os
import time

os.chdir("/content/CSAM-IPD-HALH")

if MODE in ("sweep", "both"):
    print("🚀 Starting 5-Seed Threshold Sweep...")
    t_start = time.time()

    for i, seed in enumerate(SEEDS, 1):
        out = f"{OUTPUT_DIR}/sweep_seed{seed}.json"
        print(f"\n[{i}/{len(SEEDS)}] Seed={seed}  → {out}")
        t0 = time.time()

        !python -m csam_project.evaluation.threshold_sweep \
            --thresholds 50 80 100 120 150 \
            --conversations {NUM_CONVERSATIONS} \
            --interactions {INTERACTIONS} \
            --seed {seed} \
            --output {out} \
            --model {LLM_MODEL} \
            --quiet

        elapsed = time.time() - t0
        print(f"  ✅ Seed {seed} done in {elapsed/60:.1f} min")

    total_elapsed = time.time() - t_start
    print(f"\n⏱️  All sweep seeds done in {total_elapsed/60:.1f} min")
else:
    print("⏭️  Skipping sweep (mode=ablation)")

## 9. Aggregate Results — Mean ± Std + Significance Tests

Load all per-seed JSONs, compute mean ± std per strategy, and run **Wilcoxon signed-rank tests** (CA vs each baseline).

In [ ]:
import os
import json
import numpy as np

os.chdir("/content/CSAM-IPD-HALH")

# ── Ablation Aggregation ─────────────────────────────────────────
ablation_files = [f"{OUTPUT_DIR}/ablation_seed{s}.json" for s in SEEDS]
existing_ablation = [f for f in ablation_files if os.path.exists(f)]

if existing_ablation:
    print("=" * 70)
    print(f"ABLATION: {len(existing_ablation)}/{len(SEEDS)} seeds loaded")
    print("=" * 70)

    per_seed = [json.load(open(f)) for f in existing_ablation]
    strategy_names = [r["strategy"] for r in per_seed[0]["results"]]

    agg = {}
    for name in strategy_names:
        vals = []
        for sd in per_seed:
            for r in sd["results"]:
                if r["strategy"] == name:
                    vals.append(r["overall_f1"])
                    break
        agg[name] = {
            "mean": np.mean(vals), "std": np.std(vals, ddof=1) if len(vals) > 1 else 0.0,
            "min": np.min(vals), "max": np.max(vals), "vals": vals
        }

    # Significance tests
    from scipy.stats import wilcoxon, ttest_rel
    ca_key = "Consolidation-Aware (Ours)"
    ca_vals = agg[ca_key]["vals"]

    print(f"\n{'Strategy':<30} {'Mean F1':>10} {'± Std':>8} {'Min':>8} {'Max':>8} {'p-value':>10} {'test':>10}")
    print("-" * 82)

    for name, data in agg.items():
        p_str, t_str = "", ""
        if name != ca_key:
            try:
                n = len(ca_vals)
                stat, p = (wilcoxon(ca_vals, data["vals"]) if n >= 5
                           else ttest_rel(ca_vals, data["vals"]))
                test_name = "wilcoxon" if n >= 5 else "paired-t"
                p_str = f"{p:.4f}{'*' if p < 0.05 else ''}"
                t_str = test_name
            except Exception as e:
                p_str = f"err:{e}"
        print(f"{name:<30} {data['mean']:>10.4f} "
              f"±{data['std']:>7.4f} "
              f"{data['min']:>8.4f} {data['max']:>8.4f} "
              f"{p_str:>10} {t_str:>10}")

    # CA wins across seeds
    wins = {}
    for name, data in agg.items():
        if name == ca_key:
            continue
        w = sum(1 for c, b in zip(ca_vals, data["vals"]) if c > b)
        wins[name] = f"{w}/{len(ca_vals)}"
    print(f"\nCA wins per seed (CA > baseline): {wins}")

    # Save aggregated results
    var_file = f"{OUTPUT_DIR}/variance_ablation_results.json"
    var_data = {
        "seeds": SEEDS,
        "n_seeds_completed": len(existing_ablation),
        "config": {"num_conversations": NUM_CONVERSATIONS,
                   "interactions": INTERACTIONS,
                   "forget_threshold": FORGET_THRESHOLD,
                   "llm_model": LLM_MODEL},
        "aggregated": {k: {kk: float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
                           for kk, vv in v.items()} for k, v in agg.items()}
    }
    with open(var_file, "w") as f:
        json.dump(var_data, f, indent=2)
    print(f"\n💾 Saved: {var_file}")
else:
    print("⚠️  No ablation seed files found — skipping aggregation")


# ── Sweep Aggregation ────────────────────────────────────────────
sweep_files = [f"{OUTPUT_DIR}/sweep_seed{s}.json" for s in SEEDS]
existing_sweep = [f for f in sweep_files if os.path.exists(f)]

if existing_sweep:
    print("\n" + "=" * 70)
    print(f"SWEEP: {len(existing_sweep)}/{len(SEEDS)} seeds loaded")
    print("=" * 70)

    sds = [json.load(open(f)) for f in existing_sweep]

    # No-Forgetting
    nf_vals = [s["no_forgetting"]["overall_f1"] for s in sds]
    print(f"\nNo-Forgetting: {np.mean(nf_vals):.4f} ± {np.std(nf_vals, ddof=1) if len(nf_vals)>1 else 0.0:.4f}")

    thresholds = [e["threshold"] for e in sds[0]["sweep"]]
    print(f"\n{'Threshold':<12} {'Mean F1':>10} {'± Std':>8} {'Min':>8} {'Max':>8}")
    print("-" * 46)

    sweep_agg = {}
    for thresh in thresholds:
        vals = []
        for sd in sds:
            for e in sd["sweep"]:
                if e["threshold"] == thresh:
                    vals.append(e["overall_f1"])
                    break
        sweep_agg[str(thresh)] = {"mean": np.mean(vals), "std": np.std(vals, ddof=1) if len(vals)>1 else 0.0,
                                   "min": np.min(vals), "max": np.max(vals), "vals": vals}
        print(f"{thresh:<12} {np.mean(vals):>10.4f} ±{np.std(vals, ddof=1) if len(vals)>1 else 0.0:>7.4f} "
              f"{np.min(vals):>8.4f} {np.max(vals):>8.4f}")

    var_sweep_file = f"{OUTPUT_DIR}/variance_sweep_results.json"
    with open(var_sweep_file, "w") as f:
        json.dump({"seeds": SEEDS, "no_forgetting_f1_values": nf_vals,
                   "thresholds": sweep_agg}, f, indent=2, default=float)
    print(f"\n💾 Saved: {var_sweep_file}")
else:
    print("⚠️  No sweep seed files found — skipping aggregation")

## 10. Validate Results

In [ ]:
import os
import json

os.chdir("/content/CSAM-IPD-HALH")

passed = 0
failed = 0

def check(name, condition):
    global passed, failed
    if condition:
        print(f"  ✅ {name}")
        passed += 1
    else:
        print(f"  ❌ {name}")
        failed += 1

print("=" * 60)
print("VALIDATION")
print("=" * 60)

# Check all per-seed ablation files
print("\n📋 Per-seed ablation files:")
for seed in SEEDS:
    fp = f"{OUTPUT_DIR}/ablation_seed{seed}.json"
    check(f"ablation_seed{seed}.json exists", os.path.exists(fp))

# Check aggregated
var_file = f"{OUTPUT_DIR}/variance_ablation_results.json"
check("variance_ablation_results.json exists", os.path.exists(var_file))

if os.path.exists(var_file):
    with open(var_file) as f:
        var = json.load(f)
    ca = var["aggregated"].get("Consolidation-Aware (Ours)", {})
    check(f"CA mean F1 > 0 (got {ca.get('mean', 0):.4f})", ca.get("mean", 0) > 0)
    check("CA std computed", "std" in ca)

    # CA vs LRU mean
    lru = var["aggregated"].get("LRU", {})
    ca_mean = ca.get("mean", 0)
    lru_mean = lru.get("mean", 0)
    check(f"CA mean ({ca_mean:.4f}) >= LRU mean ({lru_mean:.4f})", ca_mean >= lru_mean)

print("\n📋 Per-seed sweep files:")
for seed in SEEDS:
    fp = f"{OUTPUT_DIR}/sweep_seed{seed}.json"
    check(f"sweep_seed{seed}.json exists", os.path.exists(fp))

var_sweep_file = f"{OUTPUT_DIR}/variance_sweep_results.json"
check("variance_sweep_results.json exists", os.path.exists(var_sweep_file))

print(f"\n{'=' * 60}")
status = "PASS ✅" if failed == 0 else "FAIL ❌"
print(f"RESULT: {status} — {passed} passed, {failed} failed")
print(f"{'=' * 60}")

## 11. Download All Result Files

In [ ]:
import os
from google.colab import files

os.chdir("/content/CSAM-IPD-HALH")

# Collect all result files
result_files = []

# Per-seed ablation
for seed in SEEDS:
    result_files.append(f"{OUTPUT_DIR}/ablation_seed{seed}.json")

# Per-seed sweep
for seed in SEEDS:
    result_files.append(f"{OUTPUT_DIR}/sweep_seed{seed}.json")

# Aggregated
result_files += [
    f"{OUTPUT_DIR}/variance_ablation_results.json",
    f"{OUTPUT_DIR}/variance_sweep_results.json",
]

downloaded = 0
for fp in result_files:
    if os.path.exists(fp):
        files.download(fp)
        print(f"📥 Downloaded: {os.path.basename(fp)}")
        downloaded += 1
    else:
        print(f"⚠️  Not found: {fp}")

print(f"\nTotal: {downloaded} file(s) downloaded")